# 06 — Silver Events and Payment Status CDC

This notebook cleans authentication events and payment-status change events.

**CDC flow:** validate status events → preserve the complete event history → select the latest event per transaction → Delta MERGE the latest status into `silver_card_transactions`.

- `change_event_id` is the event key. Do not deduplicate by `transaction_id`.
- `change_sequence` and `change_ts` are used to order source events.
- Authentication events remain a separate event stream; they are not CDC.
- No invented CDC operation, effective-date, or sequence columns are added.
- Bronze tables are not modified. No `_rescued_data` or `_amount_raw` columns are added.


In [0]:
from pyspark.sql.functions import col, trim, upper, lit, lag
from pyspark.sql.window import Window

BRONZE = "capstone_team01.bronze_schema"
SILVER = "capstone_team01.silver_schema"

spark.sql("CREATE SCHEMA IF NOT EXISTS capstone_team01.silver_schema")

## Part A — Device authentication events

### 1. Read Bronze table

In [0]:
auth = spark.table(f"{BRONZE}.bronze_device_authentication_events")

print("Bronze authentication count:", auth.count())
auth.printSchema()
display(auth.limit(5))

### 2. Cast columns

In [0]:
auth = (auth
    .withColumn("auth_event_id", trim(col("auth_event_id").cast("string")))
    .withColumn("transaction_id", trim(col("transaction_id").cast("string")))
    .withColumn("customer_id", trim(col("customer_id").cast("string")))
    .withColumn("event_ts", col("event_ts").cast("timestamp"))
    .withColumn("device_id", trim(col("device_id").cast("string")))
    .withColumn("authentication_method", upper(trim(col("authentication_method").cast("string"))))
    .withColumn("authentication_result", upper(trim(col("authentication_result").cast("string"))))
    .withColumn("ip_country", upper(trim(col("ip_country").cast("string"))))
    .withColumn("attempt_number", col("attempt_number").cast("integer"))
    .withColumn("risk_signal", upper(trim(col("risk_signal").cast("string"))))
)

auth.printSchema()
display(auth.limit(5))

### 3. Validate transaction and customer references

In [0]:
# Read transaction keys from Silver
transaction_keys = spark.table(f"{SILVER}.silver_card_transactions").select(
    "transaction_id",
    col("customer_id").alias("transaction_customer_id")
).dropDuplicates(["transaction_id"])

# Check that the transaction exists and belongs to the same customer
auth = auth.join(transaction_keys, "transaction_id", "left")

auth_invalid = auth.filter(
    col("auth_event_id").isNull() |
    (col("auth_event_id") == "") |
    col("transaction_id").isNull() |
    col("transaction_customer_id").isNull() |
    col("customer_id").isNull() |
    (col("customer_id") != col("transaction_customer_id")) |
    col("event_ts").isNull() |
    col("attempt_number").isNull() |
    (col("attempt_number") <= 0)
)

auth_valid = auth.filter(
    col("auth_event_id").isNotNull() &
    (col("auth_event_id") != "") &
    col("transaction_id").isNotNull() &
    col("transaction_customer_id").isNotNull() &
    col("customer_id").isNotNull() &
    (col("customer_id") == col("transaction_customer_id")) &
    col("event_ts").isNotNull() &
    col("attempt_number").isNotNull() &
    (col("attempt_number") > 0)
)

# Remove the temporary reference column
auth_invalid = auth_invalid.drop("transaction_customer_id")
auth_valid = auth_valid.drop("transaction_customer_id")

auth_invalid = auth_invalid.withColumn(
    "dq_reason",
    lit("Missing event ID, invalid timestamp/attempt number, or transaction/customer reference mismatch")
)

print("Invalid authentication events:", auth_invalid.count())
print("Valid authentication events:", auth_valid.count())

### 4. Handle duplicate authentication event IDs

In [0]:
# Keep one record per auth_event_id
auth_silver = auth_valid.dropDuplicates(["auth_event_id"])

# Get extra duplicate records
auth_duplicates = auth_valid.exceptAll(auth_silver)

auth_duplicates = auth_duplicates.withColumn(
    "dq_reason", lit("Duplicate auth_event_id")
)

auth_quarantine = auth_invalid.unionByName(auth_duplicates)

print("Authentication Silver:", auth_silver.count())
print("Authentication Quarantine:", auth_quarantine.count())

### 5. Save tables

In [0]:
auth_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.silver_device_authentication_events")

if auth_quarantine.count() > 0:
    auth_quarantine.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{SILVER}.quarantine_device_authentication_events")
    print("Authentication quarantine table created.")
else:
    spark.sql(f"DROP TABLE IF EXISTS {SILVER}.quarantine_device_authentication_events")
    print("No rejected authentication events. Quarantine table not created.")

## Part B — Payment status changes

### 6. Read Bronze table

In [0]:
status = spark.table(f"{BRONZE}.bronze_payment_status_changes")

print("Bronze payment status count:", status.count())
status.printSchema()
display(status.limit(5))

### 7. Cast columns

In [0]:
status = (status
    .withColumn("change_event_id", trim(col("change_event_id").cast("string")))
    .withColumn("transaction_id", trim(col("transaction_id").cast("string")))
    .withColumn("change_ts", col("change_ts").cast("timestamp"))
    .withColumn("old_status", upper(trim(col("old_status").cast("string"))))
    .withColumn("new_status", upper(trim(col("new_status").cast("string"))))
    .withColumn("reason_code", trim(col("reason_code").cast("string")))
    .withColumn("change_sequence", col("change_sequence").cast("integer"))
    .withColumn("source_system", trim(col("source_system").cast("string")))
    .withColumn("operator_id", trim(col("operator_id").cast("string")))
    .withColumn("correlation_id", trim(col("correlation_id").cast("string")))
)

status.printSchema()
display(status.limit(5))

### 8. Validate transaction references and required fields

In [0]:
# Get valid transaction IDs
transaction_ids = spark.table(f"{SILVER}.silver_card_transactions").select(
    "transaction_id"
).dropDuplicates()

status = status.join(
    transaction_ids.withColumn("transaction_exists", lit(True)),
    "transaction_id",
    "left"
)

status_invalid = status.filter(
    col("change_event_id").isNull() |
    (col("change_event_id") == "") |
    col("transaction_id").isNull() |
    col("transaction_exists").isNull() |
    col("change_ts").isNull() |
    col("old_status").isNull() |
    col("new_status").isNull() |
    col("change_sequence").isNull() |
    (col("change_sequence") <= 0)
)

status_valid = status.filter(
    col("change_event_id").isNotNull() &
    (col("change_event_id") != "") &
    col("transaction_id").isNotNull() &
    col("transaction_exists").isNotNull() &
    col("change_ts").isNotNull() &
    col("old_status").isNotNull() &
    col("new_status").isNotNull() &
    col("change_sequence").isNotNull() &
    (col("change_sequence") > 0)
)

status_invalid = status_invalid.drop("transaction_exists")
status_valid = status_valid.drop("transaction_exists")

status_invalid = status_invalid.withColumn(
    "dq_reason",
    lit("Missing event ID, invalid timestamp/sequence/status, or transaction not found")
)

print("Invalid payment status rows:", status_invalid.count())
print("Valid payment status rows:", status_valid.count())

### 9. Check payment status sequence

In [0]:
# Compare each event's old_status with the previous event's new_status
status_window = Window.partitionBy("transaction_id").orderBy("change_sequence")

status_sequence = status_valid.withColumn(
    "previous_new_status",
    lag("new_status").over(status_window)
)

# A mismatch is checked only when a previous event exists
status_sequence_invalid = status_sequence.filter(
    col("previous_new_status").isNotNull() &
    (col("old_status") != col("previous_new_status"))
)

status_sequence_valid = status_sequence.filter(
    col("previous_new_status").isNull() |
    (col("old_status") == col("previous_new_status"))
)

status_sequence_invalid = status_sequence_invalid.drop("previous_new_status")
status_sequence_valid = status_sequence_valid.drop("previous_new_status")

status_sequence_invalid = status_sequence_invalid.withColumn(
    "dq_reason",
    lit("Payment status sequence does not match the previous event")
)

# Combine basic invalid rows and sequence-invalid rows
status_invalid = status_invalid.unionByName(status_sequence_invalid)

print("Payment status rows after sequence check:", status_sequence_valid.count())
print("Payment status rows rejected:", status_invalid.count())

### 10. Handle duplicate payment event IDs

In [0]:
# Keep one row per change_event_id, not one row per transaction_id
status_silver = status_sequence_valid.dropDuplicates(["change_event_id"])

# Get extra duplicate event records
status_duplicates = status_sequence_valid.exceptAll(status_silver)

status_duplicates = status_duplicates.withColumn(
    "dq_reason", lit("Duplicate change_event_id")
)

status_quarantine = status_invalid.unionByName(status_duplicates)

print("Payment status Silver:", status_silver.count())
print("Payment status Quarantine:", status_quarantine.count())

### 11. Save payment status tables

In [0]:
status_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SILVER}.silver_payment_status_changes")

if status_quarantine.count() > 0:
    status_quarantine.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(f"{SILVER}.quarantine_payment_status_changes")
    print("Payment status quarantine table created.")
else:
    spark.sql(f"DROP TABLE IF EXISTS {SILVER}.quarantine_payment_status_changes")
    print("No rejected payment status rows. Quarantine table not created.")

## 12. Apply payment-status CDC with Delta MERGE

The Silver status table above preserves every accepted status-change event. The next cells apply only the latest event for each transaction to the current transaction record. This keeps event history and current state separate.

### 12.1 Select the latest status event per transaction

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Order by the source-provided sequence, then timestamp as a tie-breaker.
latest_window = Window.partitionBy("transaction_id").orderBy(
    col("change_sequence").desc(),
    col("change_ts").desc(),
    col("change_event_id").desc()
)

latest_status = (
    spark.table(f"{SILVER}.silver_payment_status_changes")
    .withColumn("row_num", row_number().over(latest_window))
    .filter(col("row_num") == 1)
    .select(
        "transaction_id",
        col("new_status").alias("latest_payment_status"),
        col("change_sequence").alias("latest_change_sequence"),
        col("change_ts").alias("latest_change_ts"),
        "change_event_id"
    )
)

print("Transactions with CDC status updates:", latest_status.count())
display(latest_status.orderBy("transaction_id"))


### 12.2 Validate the CDC source before MERGE

In [0]:
# Every CDC transaction must exist in the current Silver transaction table.
transaction_ids = spark.table(f"{SILVER}.silver_card_transactions").select("transaction_id").dropDuplicates()

cdc_missing_transactions = latest_status.join(
    transaction_ids,
    "transaction_id",
    "left_anti"
)

print("CDC rows without a Silver transaction:", cdc_missing_transactions.count())
display(cdc_missing_transactions)


### 12.3 MERGE latest status into the transaction table

In [0]:
# Keep only CDC records whose transaction exists in Silver.
latest_status_for_merge = latest_status.join(
    transaction_ids,
    "transaction_id",
    "inner"
)

latest_status_for_merge.createOrReplaceTempView("latest_payment_status_for_merge")

# Idempotent for a full-history replay: the same latest status is not repeatedly rewritten.
spark.sql(f"""
MERGE INTO {SILVER}.silver_card_transactions AS target
USING latest_payment_status_for_merge AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED
 AND NOT (target.payment_status <=> source.latest_payment_status)
THEN UPDATE SET
    target.payment_status = source.latest_payment_status
""")

print("Latest payment status MERGE completed.")


### 12.4 Validate the applied current status

In [0]:
current_status_check = (
    spark.table(f"{SILVER}.silver_card_transactions").alias("t")
    .join(latest_status_for_merge.alias("s"), "transaction_id", "inner")
    .select(
        "transaction_id",
        col("t.payment_status").alias("current_payment_status"),
        col("s.latest_payment_status")
    )
)

cdc_mismatch = current_status_check.filter(
    col("current_payment_status") != col("latest_payment_status")
)

print("CDC status mismatches:", cdc_mismatch.count())
display(cdc_mismatch)


**Important:** This notebook applies the latest status represented by the delivered `payment_status_changes` history. It does not invent CDC operation types or effective dates. If the pipeline later receives incremental CDC batches rather than replaying the complete history, persist and compare the last applied `change_sequence` (or use a dedicated current-state table with that sequence) to prevent an older batch from overwriting a newer state.

## 13. Final validation

## 14. Final validation


In [0]:
# Authentication reconciliation
auth_bronze_count = spark.table(f"{BRONZE}.bronze_device_authentication_events").count()
auth_silver_count = spark.table(f"{SILVER}.silver_device_authentication_events").count()

if spark.catalog.tableExists(f"{SILVER}.quarantine_device_authentication_events"):
    auth_quarantine_count = spark.table(f"{SILVER}.quarantine_device_authentication_events").count()
else:
    auth_quarantine_count = 0

print("Authentication Bronze:", auth_bronze_count)
print("Authentication Silver:", auth_silver_count)
print("Authentication Quarantine:", auth_quarantine_count)
print("Authentication reconciled:", auth_bronze_count == auth_silver_count + auth_quarantine_count)

# Payment status reconciliation
status_bronze_count = spark.table(f"{BRONZE}.bronze_payment_status_changes").count()
status_silver_count = spark.table(f"{SILVER}.silver_payment_status_changes").count()

if spark.catalog.tableExists(f"{SILVER}.quarantine_payment_status_changes"):
    status_quarantine_count = spark.table(f"{SILVER}.quarantine_payment_status_changes").count()
else:
    status_quarantine_count = 0

print("Payment status Bronze:", status_bronze_count)
print("Payment status Silver:", status_silver_count)
print("Payment status Quarantine:", status_quarantine_count)
print("Payment status reconciled:", status_bronze_count == status_silver_count + status_quarantine_count)

# Check duplicate event IDs in Silver
display(
    spark.table(f"{SILVER}.silver_device_authentication_events")
    .groupBy("auth_event_id").count().filter(col("count") > 1)
)

display(
    spark.table(f"{SILVER}.silver_payment_status_changes")
    .groupBy("change_event_id").count().filter(col("count") > 1)
)

In [0]:
from pyspark.sql.functions import col

# Flag events whose source-provided risk_signal is not NONE
auth = spark.table("capstone_team01.silver_schema.silver_device_authentication_events").withColumn(
    "risk_review_flag",
    col("risk_signal").isNotNull() & (col("risk_signal") != "NONE")
)

# Display events containing source-provided risk signals

display(
    auth.filter(
        col("risk_review_flag") == True
    ).select(
        "auth_event_id",
        "transaction_id",
        "risk_signal",
        "risk_review_flag"
    )
)